# Who does the village vote for?

**Question.** The same one the LLM surrogate asks, with a different voter: how
much of the village's vote can be reconstructed from interpretable features of
the dialogue, and does it matter *who a persuasion act was aimed at and what it
claimed* or only that the act occurred?

**Why this notebook exists.** The village's votes have the same shape as an
LLM's: on each game, *n* voters distribute their votes over the roster. That is
a choice set with counts, so the same conditional logit applies, and the human
and LLM results can be compared without two different modelling pipelines
standing between them. The machinery is shared: `src/utils_choice`.

It also replaces the candidate-only models that used to sit in the replication
notebook. Those were ill-posed -- with features describing only the candidate,
every (voter, candidate) row for a given candidate in a game is identical while
the labels differ by voter, so the best any model could do was recover that
candidate's vote share, and the pairwise framing added only duplicated,
correlated rows. Modelling the vote distribution directly is the well-posed
version of the same question.

**What this comparison is, and is not.**

- **The LLMs are not players.** They read a finished transcript and name a
  suspect; the humans were in the game, acting under social pressure and with
  private role knowledge. The question here is narrow: do the same dialogue
  features explain both sets of votes? It is not a claim that LLMs behave like
  players.
- **The ceilings mean different things.** For an LLM, agreement across runs is
  test-retest reliability of one agent. For the village it is agreement between
  *different people*. Both bound how well any model can do; they are not the
  same construct.
- **The ballots differ.** Human voters here always name a player, so the human
  choice set is the roster alone, with no abstention alternative. Pseudo-R² is
  normalised by each side's own null model, which is why it stays the right
  quantity to compare.

Only **village-aligned voters** are counted -- werewolves vote strategically,
so their votes do not measure suspicion -- and games with fewer than two such
voters cannot express a distribution and are dropped.

## Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(repo_name)
        current = current.parent


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))

from utils_choice import (BLOCKS, BLOCK_A, BLOCK_B, block_bootstrap, block_test,
                          build_ballot, build_frame, coef_table, cols_for,
                          cross_validate, human_vote_targets,
                          load_accusation_features, load_identity_claim_features,
                          load_technique_features, load_vote_tables,
                          reference_points, run_grid, run_validation_checks,
                          save_tables, shares_from_targets, stability_selection,
                          STABILITY_THRESHOLD)

PROMPT_DIR = "prompt_v4"
# which stage of the models this analysis covers
MODEL_STAGE = "base"
ANALYSIS_ROOT = REPO_ROOT / "analysis"
TABLES_REL = Path("base") / "voting" / PROMPT_DIR / "vote_stability" / "tables"
ANNOT_ROOT = REPO_ROOT / "data" / "raw" / "lai2023"
ACC_ROOT = (REPO_ROOT / "data" / "processed" / "lai2023"
            / "accusation_transcripts" / "acc_targets")
IC_CSV = (REPO_ROOT / "data" / "processed" / "lai2023"
          / "identity_claim_transcripts" / "ic_targets"
          / "player_conflict_features.csv")
# The prompt directory qualifies the LLM side only -- it names the voting
# prompt the models were given. Nothing about human votes is filed under one.
VILLAGE_CSV = ANALYSIS_ROOT / "human_outcomes" / "vote_tables" / "village_vote_dispersion.csv"
LLM_TABLES = ANALYSIS_ROOT / "cross_model" / MODEL_STAGE / "voting" / PROMPT_DIR / "predictive" / "tables"
OUT_DIR = ANALYSIS_ROOT / "human_outcomes" / "choice_model" / "tables"

for msg in run_validation_checks():
    print("estimator validated:", msg)

estimator validated: recovers a known beta from simulated choices (max |error| = 0.018)
estimator validated: matches scikit-learn on the J=2 reduction (max |difference| = 1.6e-04)
estimator validated: analytic gradient matches the numerical one (error = 5.2e-05)
estimator validated: choice probabilities sum to 1 in every set (max error = 2.2e-16)
estimator validated: a strong L1 penalty zeroes coefficients (1 of 6 survive)


## Features, ballot and target

Identical feature construction to the LLM notebook -- the same two blocks and
their temporal variants -- so any difference in the results is a difference in
the voter, not in the pipeline.

In [2]:
votes, games, roster_by_key = load_vote_tables(ANALYSIS_ROOT, TABLES_REL)
pt_df, game_length, rec_speaker, annotated = load_technique_features(ANNOT_ROOT)
acc_counts, n_acc = load_accusation_features(ACC_ROOT, game_length, rec_speaker)
ic_feats = load_identity_claim_features(IC_CSV, game_length, rec_speaker)
ballot = build_ballot(roster_by_key, pt_df, acc_counts, ic_feats)

TARGETS, summary, unmatched, village = human_vote_targets(VILLAGE_CSV, roster_by_key)
frame = build_frame(TARGETS, ballot, include_circle=False)
shares = shares_from_targets(TARGETS)

print(f"human choice sets: {frame['key'].nunique()} games, "
      f"{frame['count'].sum():.0f} village votes "
      f"({unmatched} dropped for unmatched names)")
print(f"games where the village split its vote: {summary['is_split'].mean():.1%}; "
      f"mean voters per game: {summary['n_voters'].mean():.2f}")

human choice sets: 169 games, 481 village votes (0 dropped for unmatched names)
games where the village split its vote: 57.4%; mean voters per game: 2.85


## Results

In [3]:
results = run_grid(frame, BLOCKS, include_circle=False, label="human_village")
references = reference_points(frame, shares, label="human_village")

print("McFadden pseudo-R^2 (conditional logit only; gbm is not a choice model):")
display(results.pivot_table(index="block", columns="learner",
                            values="mcfadden_r2").round(3))
print("\nHit rate:")
display(results.pivot_table(index="block", columns="learner",
                            values="hit_rate").round(3))
print("\nReference points (ceiling_agreement here is agreement between "
      "different voters, not one agent resampled):")
display(references.round(3))

McFadden pseudo-R^2 (conditional logit only; gbm is not a choice model):


learner,clogit,clogit_l1,clogit_l2
block,,,
A_techniques,0.009,0.011,0.013
A_temporal,-0.014,0.004,0.002
B_resolved,0.068,0.068,0.067
B_temporal,0.058,0.059,0.061
C_both,0.065,0.073,0.071
C_temporal,0.031,0.060,0.059



Hit rate:


learner,clogit,clogit_l1,clogit_l2,gbm
block,,,,
A_techniques,0.251,0.249,0.242,0.235
A_temporal,0.224,0.209,0.223,0.229
B_resolved,0.381,0.391,0.381,0.399
B_temporal,0.387,0.388,0.387,0.352
C_both,0.370,0.391,0.368,0.391
C_temporal,0.351,0.377,0.358,0.312



Reference points (ceiling_agreement here is agreement between different voters, not one agent resampled):


,model,null_ll,null_hit_rate,ceiling_agreement,ceiling_best_possible,irreducible_entropy,most_talkative_hit,most_accused_hit
0,human_village,-1.535,0.219,0.703,0.737,0.454,0.194,0.374


## The pre-specified model

The same fixed choice as the LLM notebook: the full feature set `C_both` with
the ridge penalty, decided in advance so the coefficients and their intervals
carry no selection bias. Coefficients are log-odds within a ballot -- "relative
to the other players in this game".

In [4]:
PRIMARY_BLOCK, PRIMARY_LEARNER = "C_both", "clogit_l2"
PRIMARY_COLS = cols_for(PRIMARY_BLOCK, include_circle=False)

grid = (results[results["learner"] != "gbm"]
        .sort_values("mcfadden_r2", ascending=False).reset_index(drop=True))
hit = grid[(grid["block"] == PRIMARY_BLOCK) & (grid["learner"] == PRIMARY_LEARNER)]
best_config = pd.DataFrame([{
    "model": "human_village", "block": PRIMARY_BLOCK, "learner": PRIMARY_LEARNER,
    "mcfadden_r2": hit.iloc[0]["mcfadden_r2"], "hit_rate": hit.iloc[0]["hit_rate"],
    "rank_in_grid": f"{int(hit.index[0]) + 1} of {len(grid)}",
    "cost_of_prespecifying": round(grid.iloc[0]["mcfadden_r2"]
                                   - hit.iloc[0]["mcfadden_r2"], 3)}])
display(best_config)

coefficients, _, n_clusters = coef_table(frame, PRIMARY_COLS)
coefficients.insert(0, "model", "human_village")
print(f"\nOdds ratios per 1 SD, cluster-robust by game ({n_clusters} games):")
display(coefficients.drop(columns="model").sort_values("p")
        .round({"odds_ratio": 2, "ci_lo": 2, "ci_hi": 2, "z": 2,
                "p": 4, "p_bonferroni": 3}).reset_index(drop=True))

_, perm = cross_validate(frame, PRIMARY_COLS, PRIMARY_LEARNER, collect_perm=True)
permutation_importance = pd.DataFrame(
    [{"model": "human_village", "feature": f,
      "ll_drop": round(float(np.mean(d)), 4),
      "share_folds_positive": round(float(np.mean(np.array(d) > 0)), 2)}
     for f, d in perm.items()])
print("\nHeld-out permutation importance:")
display(permutation_importance.sort_values("ll_drop", ascending=False)
        .drop(columns="model").reset_index(drop=True).head(10))

,model,block,learner,mcfadden_r2,hit_rate,rank_in_grid,cost_of_prespecifying
0,human_village,C_both,clogit_l2,0.071,0.368,2 of 18,0.002



Odds ratios per 1 SD, cluster-robust by game (169 games):


,feature,odds_ratio,ci_lo,ci_hi,z,p,p_bonferroni
0,werewolf_count,1.71,1.42,2.06,5.66,0.0000,0.000
1,claims_info_role,0.78,0.67,0.92,-2.96,0.0031,0.037
2,pt_evidence,0.78,0.65,0.92,-2.93,0.0034,0.040
3,pt_call_for_action,1.26,1.04,1.53,2.35,0.0186,0.223
4,n_distinct_roles_claimed_self,1.20,0.94,1.55,1.46,0.1443,1.000
5,claims_werewolf,1.08,0.91,1.28,0.87,0.3846,1.000
6,pt_accusation,1.08,0.86,1.37,0.67,0.5032,1.000
7,pt_defense,1.08,0.84,1.40,0.60,0.5467,1.000
8,n_utterances,0.89,0.60,1.32,-0.60,0.5503,1.000
9,pt_interrogation,0.94,0.77,1.16,-0.58,0.5641,1.000



Held-out permutation importance:


,feature,ll_drop,share_folds_positive
0,werewolf_count,0.1391,1.00
1,pt_evidence,0.0397,0.93
2,claims_info_role,0.0240,0.87
3,pt_call_for_action,0.0123,0.73
4,claims_werewolf,0.0048,0.80
5,n_distinct_roles_claimed_self,0.0036,0.67
6,pt_identity_declaration,0.0014,0.60
7,pt_defense,0.0010,0.53
8,pt_interrogation,0.0006,0.67
9,pt_accusation,-0.0000,0.40


## Inference

Same three tools as the LLM notebook: robust Wald tests for what one block adds
to another, stability selection for which features are worth interpreting, and
a game-level bootstrap on the out-of-sample differences.

The IIA check has no counterpart here: it works by dropping the abstention
alternative, and the human ballot has none.

In [5]:
block_rows = []
for small, label in [("A_techniques", "B_resolved adds over A_techniques"),
                     ("B_resolved", "A_techniques adds over B_resolved")]:
    W, dfree, p = block_test(frame, small, include_circle=False)
    block_rows.append({"model": "human_village", "test": label,
                       "chi2": round(W, 1), "df": dfree, "p_value": p})
block_tests = pd.DataFrame(block_rows)

stability, _ = stability_selection(frame, PRIMARY_COLS)
stability.insert(0, "model", "human_village")

bootstrap = block_bootstrap(
    frame, [("A_techniques", "B_resolved"), ("B_resolved", "C_both"),
            ("A_techniques", "C_both"), ("A_techniques", "A_temporal"),
            ("B_resolved", "B_temporal")],
    null_ll=-references["null_ll"].iloc[0], include_circle=False,
    label="human_village")

print("Nested block tests (robust Wald):")
display(block_tests)
print(f"\nStability selection (>= {STABILITY_THRESHOLD} is the conventional threshold):")
display(stability.sort_values("selection_freq", ascending=False)
        .drop(columns="model").reset_index(drop=True).head(10))
print("\nGame-level bootstrap on held-out log-likelihood:")
display(bootstrap)

Nested block tests (robust Wald):


,model,test,chi2,df,p_value
0,human_village,B_resolved adds over A_techniques,49.2,5,2.021019e-09
1,human_village,A_techniques adds over B_resolved,17.6,7,1.410297e-02



Stability selection (>= 0.6 is the conventional threshold):


,feature,selection_freq,lambda
0,werewolf_count,1.000,27.029
1,pt_evidence,0.790,27.029
2,claims_info_role,0.635,27.029
3,claims_werewolf,0.555,27.029
4,pt_call_for_action,0.185,27.029
5,n_distinct_roles_claimed_self,0.075,27.029
6,pt_interrogation,0.055,27.029
7,pt_identity_declaration,0.035,27.029
8,n_utterances,0.005,27.029
9,deception_count,0.005,27.029



Game-level bootstrap on held-out log-likelihood:


,model,comparison,delta_mean_ll,ci_lo,ci_hi,delta_pseudo_r2,excludes_zero
0,human_village,B_resolved - A_techniques,0.0941,0.0429,0.1452,0.0613,True
1,human_village,C_both - B_resolved,0.0066,-0.0122,0.0253,0.0043,False
2,human_village,C_both - A_techniques,0.1008,0.0603,0.1418,0.0656,True
3,human_village,A_temporal - A_techniques,-0.0171,-0.0273,-0.0082,-0.0112,True
4,human_village,B_temporal - B_resolved,-0.0089,-0.0255,0.0075,-0.0058,False


## Village vs LLMs

Same estimator, same features, same cross-validation; only the voter changes.
Read pseudo-R² across rows (it is normalised by each side's own null) and read
the hit rate against each side's own ceiling, never against 1.0.

In [6]:
llm_results = pd.read_csv(LLM_TABLES / "surrogate_results.csv")
llm_refs = pd.read_csv(LLM_TABLES / "surrogate_reference_points.csv")
llm_coefs = pd.read_csv(LLM_TABLES / "surrogate_coefficients.csv")

r2 = (pd.concat([results, llm_results])
      .query("learner == @PRIMARY_LEARNER")
      .pivot_table(index="model", columns="block", values="mcfadden_r2")
      [["A_techniques", "B_resolved", "C_both"]])
print("McFadden pseudo-R^2 (ridge, identical features):")
display(r2.round(3))

ceilings = pd.concat([references, llm_refs])[
    ["model", "null_hit_rate", "ceiling_agreement"]].set_index("model")
hits = (pd.concat([results, llm_results]).query(
    "learner == @PRIMARY_LEARNER and block == @PRIMARY_BLOCK")
    .set_index("model")["hit_rate"])
reach = ceilings.join(hits)
reach["share_of_reachable"] = ((reach["hit_rate"] - reach["null_hit_rate"])
                               / (reach["ceiling_agreement"] - reach["null_hit_rate"]))
print("\nHit rate against each voter's own null and ceiling:")
display(reach.round(3))

odds = (pd.concat([coefficients, llm_coefs])
        .pivot_table(index="feature", columns="model", values="odds_ratio"))
sig = (pd.concat([coefficients, llm_coefs])
       .pivot_table(index="feature", columns="model", values="p_bonferroni"))
print("\nOdds ratios per 1 SD (* = Bonferroni-significant within that model):")
display(odds.loc[[c for c in PRIMARY_COLS if c in odds.index]].round(2)
        .astype(str).where(sig.loc[[c for c in PRIMARY_COLS if c in sig.index]] >= 0.05,
                           odds.round(2).astype(str) + "*"))
comparison = reach.reset_index().merge(r2.reset_index(), on="model")

McFadden pseudo-R^2 (ridge, identical features):


block,A_techniques,B_resolved,C_both
model,,,
2B,0.011,0.180,0.189
31B,0.011,0.160,0.160
4B,0.049,0.211,0.196
human_village,0.013,0.067,0.071



Hit rate against each voter's own null and ceiling:


,null_hit_rate,ceiling_agreement,hit_rate,share_of_reachable
model,,,,
human_village,0.219,0.703,0.368,0.308
2B,0.185,0.741,0.495,0.558
31B,0.184,0.815,0.433,0.394
4B,0.184,0.773,0.473,0.490



Odds ratios per 1 SD (* = Bonferroni-significant within that model):


model,2B,31B,4B,human_village
feature,,,,
pt_accusation,1.36,1.25,1.06,1.08
pt_defense,1.46*,1.28,1.21,1.08
pt_interrogation,1.12,0.97,0.95,0.94
pt_identity_declaration,0.88,0.87,1.01,0.97
pt_evidence,0.95,0.92,0.94,0.78*
pt_call_for_action,0.97,1.18,0.99,1.26
n_utterances,0.48*,0.67,0.96,0.89
werewolf_count,2.17*,2.06*,1.64*,1.71*
deception_count,1.05,1.34*,1.21,1.03


In [7]:
written, removed = save_tables(OUT_DIR, {
    "human_results": results,
    "human_reference_points": references,
    "human_best_config": best_config,
    "human_coefficients": coefficients,
    "human_permutation_importance": permutation_importance,
    "human_block_tests": block_tests,
    "human_stability_selection": stability,
    "human_block_bootstrap": bootstrap,
    "human_vs_llm_comparison": comparison,
})
print("saved ->", OUT_DIR.relative_to(REPO_ROOT))
for n in written:
    print("  ", n)
if removed:
    print("removed stale files:", removed)

saved -> analysis\human_outcomes\choice_model\tables
   human_best_config.csv
   human_block_bootstrap.csv
   human_block_tests.csv
   human_coefficients.csv
   human_permutation_importance.csv
   human_reference_points.csv
   human_results.csv
   human_stability_selection.csv
   human_vs_llm_comparison.csv


## Reading guide

- The village's votes are far **less** explained by these features than the
  LLMs' are. Part of that gap is a lower ceiling -- different people disagree
  more than one model resampled does -- so quote both the pseudo-R² and the
  share-of-reachable-range figures, not just the first.
- The **block ordering is the finding**: whether resolving a persuasion act
  into its target and content beats knowing the act occurred, for humans as
  well as for LLMs.
- Where humans and LLMs differ in *which* features matter is more interesting
  than the overall gap, and is the part that connects to the justification
  analysis.
- Everything is correlational, n is under 200 games, and the LLMs were never
  players. This compares what explains two sets of votes; it does not establish
  that one process resembles the other.